# 01 — Exploratory Data Analysis

**Project:** Customer Complaint Root-Cause Classifier

**Purpose:** Inspect the raw dataset, assess data quality, understand distributions, detect synthetic template patterns, and generate visualizations.

**Dataset:** `data/raw/customer_complaints.csv` (100,000 synthetic records)

---

## 0. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from collections import Counter
import re
import warnings
import os

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

print("Libraries imported successfully.")

## 1. Load Dataset

In [ ]:
# Adjust path if running notebook from the notebooks/ folder
CSV_PATH = os.path.join("..", "data", "raw", "customer_complaints.csv")

if not os.path.exists(CSV_PATH):
    # Fallback: running from project root
    CSV_PATH = os.path.join("data", "raw", "customer_complaints.csv")

df = pd.read_csv(CSV_PATH)
print(f"Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

In [ ]:
df.head(3)

In [ ]:
df.info()

## 2. Column Validation

In [ ]:
REQUIRED_COLUMNS = [
    "complaint_id", "date_received", "product", "sub_product",
    "issue", "sub_issue", "complaint_narrative", "company",
    "channel", "state", "sentiment_label", "resolution_status"
]

present = [c for c in REQUIRED_COLUMNS if c in df.columns]
missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]

print(f"Required columns present: {len(present)}/{len(REQUIRED_COLUMNS)}")
if missing:
    print(f"MISSING: {missing}")
else:
    print("All required columns are present.")

## 3. Missing Values

In [ ]:
missing_counts = df.isnull().sum()
missing_pct = (missing_counts / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    "Column": missing_counts.index,
    "Missing Count": missing_counts.values,
    "Missing %": missing_pct.values
})

print(missing_df.to_string(index=False))
print(f"\nTotal missing values: {missing_counts.sum()}")

## 4. Duplicate Complaint IDs

In [ ]:
dup_ids = df["complaint_id"].duplicated().sum()
print(f"Duplicate complaint_id count: {dup_ids}")

if dup_ids > 0:
    dup_examples = df[df["complaint_id"].duplicated(keep=False)]
    print("\nExample duplicate IDs:")
    print(dup_examples.head(6))
else:
    print("No duplicate IDs. Each complaint has a unique identifier.")

## 5. Exact and Near-Duplicate Narratives

In [ ]:
total = len(df)
unique_narr = df["complaint_narrative"].nunique()
exact_dups = df["complaint_narrative"].duplicated().sum()

print(f"Total complaints:          {total:,}")
print(f"Unique narratives:         {unique_narr:,}")
print(f"Exact duplicate narratives: {exact_dups:,} ({exact_dups/total*100:.1f}%)")

In [ ]:
# Top duplicated narratives — how many times does each appear?
dup_counts = (
    df.groupby("complaint_narrative")
    .size()
    .sort_values(ascending=False)
    .reset_index(name="occurrences")
)

# Distribution of duplicate frequency
dup_freq = dup_counts["occurrences"].value_counts().sort_index()
print("Duplicate frequency distribution:")
print(dup_freq.to_string())

print(f"\nNarratives appearing only once: {(dup_counts['occurrences'] == 1).sum():,}")
print(f"Narratives appearing 2+ times:  {(dup_counts['occurrences'] >= 2).sum():,}")

In [ ]:
# Show the top 5 most-duplicated narratives
print("Top 5 most-repeated narratives:\n")
for idx, row in dup_counts.head(5).iterrows():
    print(f"[{row['occurrences']}x] {row['complaint_narrative'][:200]}...")
    print()

## 6. Total Complaint Count and Basic Statistics

In [ ]:
print("=" * 50)
print("DATASET OVERVIEW")
print("=" * 50)
print(f"Total complaints:      {len(df):,}")
print(f"Unique complaint IDs:  {df['complaint_id'].nunique():,}")
print(f"Unique products:       {df['product'].nunique()}")
print(f"Unique sub-products:   {df['sub_product'].nunique()}")
print(f"Unique issues:         {df['issue'].nunique()}")
print(f"Unique sub-issues:     {df['sub_issue'].nunique()}")
print(f"Unique companies:      {df['company'].nunique()}")
print(f"Unique channels:       {df['channel'].nunique()}")
print(f"Unique states:         {df['state'].nunique()}")
print(f"Sentiment labels:      {df['sentiment_label'].nunique()}")
print(f"Resolution statuses:   {df['resolution_status'].nunique()}")

## 7. Top 10 Issue Categories

In [ ]:
issue_counts = df["issue"].value_counts()
print("Issue distribution (all categories):")
print(issue_counts.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
top10_issues = issue_counts.head(10)
colors = sns.color_palette("viridis", n_colors=10)
bars = ax.barh(top10_issues.index[::-1], top10_issues.values[::-1], color=colors)

for bar, val in zip(bars, top10_issues.values[::-1]):
    ax.text(bar.get_width() + 100, bar.get_y() + bar.get_height() / 2,
            f"{val:,}", va="center", fontsize=10)

ax.set_xlabel("Number of Complaints")
ax.set_title("Top 10 Issue Categories")
plt.tight_layout()
plt.show()

## 8. Sub-Issue Distribution

In [ ]:
sub_issue_counts = df["sub_issue"].value_counts()
print(f"Total unique sub-issues: {len(sub_issue_counts)}")
print("\nSub-issue distribution:")
print(sub_issue_counts.to_string())

In [ ]:
fig = px.bar(
    x=sub_issue_counts.values,
    y=sub_issue_counts.index,
    orientation="h",
    labels={"x": "Count", "y": "Sub-Issue"},
    title="Sub-Issue Distribution (All Categories)",
    color=sub_issue_counts.values,
    color_continuous_scale="Viridis"
)
fig.update_layout(height=800, yaxis={"autorange": "reversed"})
fig.show()

## 9. Product-Wise Complaints

In [ ]:
product_counts = df["product"].value_counts()
print("Product distribution:")
print(product_counts.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
sns.barplot(x=product_counts.values, y=product_counts.index, ax=axes[0], palette="coolwarm")
axes[0].set_xlabel("Count")
axes[0].set_title("Complaints by Product")

# Pie chart
axes[1].pie(product_counts.values, labels=product_counts.index,
            autopct="%1.1f%%", startangle=140, textprops={"fontsize": 8})
axes[1].set_title("Product Share")

plt.tight_layout()
plt.show()

## 10. Company-Wise Complaints

In [ ]:
company_counts = df["company"].value_counts()
print("Company distribution:")
print(company_counts.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x=company_counts.values, y=company_counts.index, ax=ax, palette="Set2")

for i, val in enumerate(company_counts.values):
    ax.text(val + 50, i, f"{val:,}", va="center", fontsize=10)

ax.set_xlabel("Number of Complaints")
ax.set_title("Complaints by Company")
plt.tight_layout()
plt.show()

# Observation
print(f"\nCompanies are nearly uniformly distributed ({company_counts.min():,} – {company_counts.max():,}).")
print("This is a synthetic-data artifact — real data would show skewed distributions.")

## 11. Sentiment Distribution

In [ ]:
sentiment_counts = df["sentiment_label"].value_counts()
print("Sentiment distribution:")
for label, count in sentiment_counts.items():
    print(f"  {label:10s}: {count:,} ({count/len(df)*100:.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar
color_map = {"Negative": "#e74c3c", "Neutral": "#f39c12", "Positive": "#27ae60"}
bar_colors = [color_map.get(l, "gray") for l in sentiment_counts.index]
axes[0].bar(sentiment_counts.index, sentiment_counts.values, color=bar_colors)
axes[0].set_title("Sentiment Distribution")
axes[0].set_ylabel("Count")

for i, (label, val) in enumerate(sentiment_counts.items()):
    axes[0].text(i, val + 500, f"{val:,}", ha="center", fontsize=10)

# Pie
axes[1].pie(sentiment_counts.values, labels=sentiment_counts.index,
            autopct="%1.1f%%", colors=bar_colors, startangle=140)
axes[1].set_title("Sentiment Share")

plt.tight_layout()
plt.show()

print("\nNote: Dataset is imbalanced — Negative dominates at 64.1%.")
print("Consider stratified splitting and class-weighted models.")

## 12. Resolution Status

In [ ]:
resolution_counts = df["resolution_status"].value_counts()
print("Resolution status distribution:")
for label, count in resolution_counts.items():
    print(f"  {label:25s}: {count:,} ({count/len(df)*100:.1f}%)")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
res_colors = sns.color_palette("RdYlGn", n_colors=len(resolution_counts))
ax.bar(resolution_counts.index, resolution_counts.values, color=res_colors)
ax.set_title("Resolution Status Distribution")
ax.set_ylabel("Count")

for i, val in enumerate(resolution_counts.values):
    ax.text(i, val + 300, f"{val:,}", ha="center", fontsize=10)

plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 13. Complaint Date Distribution

In [ ]:
df["date_received"] = pd.to_datetime(df["date_received"], errors="coerce")

print(f"Date range: {df['date_received'].min().date()} to {df['date_received'].max().date()}")
print(f"Invalid dates: {df['date_received'].isnull().sum()}")

In [ ]:
# Monthly complaint volume
monthly = df.set_index("date_received").resample("ME").size().reset_index(name="complaints")

fig = px.line(
    monthly, x="date_received", y="complaints",
    title="Monthly Complaint Volume (2023–2025)",
    labels={"date_received": "Month", "complaints": "Number of Complaints"}
)
fig.update_traces(line_color="#3498db")
fig.show()

In [ ]:
# Yearly breakdown
df["year"] = df["date_received"].dt.year
yearly = df["year"].value_counts().sort_index()
print("Complaints per year:")
for year, count in yearly.items():
    print(f"  {year}: {count:,}")

In [ ]:
# Day-of-week distribution
df["day_of_week"] = df["date_received"].dt.day_name()
dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
dow_counts = df["day_of_week"].value_counts().reindex(dow_order)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(dow_counts.index, dow_counts.values, color=sns.color_palette("husl", 7))
ax.set_title("Complaints by Day of Week")
ax.set_ylabel("Count")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 14. Narrative Length Distribution

In [ ]:
df["narrative_length"] = df["complaint_narrative"].str.len()
df["narrative_word_count"] = df["complaint_narrative"].str.split().str.len()

print("Character length statistics:")
print(df["narrative_length"].describe().round(1))
print(f"\nWord count statistics:")
print(df["narrative_word_count"].describe().round(1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df["narrative_length"], bins=50, color="#3498db", edgecolor="white", alpha=0.8)
axes[0].set_title("Narrative Character Length Distribution")
axes[0].set_xlabel("Character Length")
axes[0].set_ylabel("Frequency")
axes[0].axvline(df["narrative_length"].mean(), color="red", linestyle="--", label=f"Mean: {df['narrative_length'].mean():.0f}")
axes[0].legend()

axes[1].hist(df["narrative_word_count"], bins=50, color="#e67e22", edgecolor="white", alpha=0.8)
axes[1].set_title("Narrative Word Count Distribution")
axes[1].set_xlabel("Word Count")
axes[1].set_ylabel("Frequency")
axes[1].axvline(df["narrative_word_count"].mean(), color="red", linestyle="--", label=f"Mean: {df['narrative_word_count'].mean():.0f}")
axes[1].legend()

plt.tight_layout()
plt.show()

print("\nSynthetic data observation:")
print(f"Narrative length range is very narrow ({df['narrative_length'].min()}–{df['narrative_length'].max()} chars).")
print("Real-world complaints typically have much higher variance (10–5000+ chars).")

## 15. Repeated Synthetic Text Templates

In [ ]:
# Detect common opening phrases (first 8 words)
first_phrases = df["complaint_narrative"].str.split().str[:8].str.join(" ")
phrase_counts = first_phrases.value_counts()

print("Top 15 opening phrases (first 8 words):")
print("-" * 70)
for phrase, cnt in phrase_counts.head(15).items():
    print(f"  [{cnt:5,}] {phrase}")

In [ ]:
# Detect shared filler / template sentences
all_sentences = []
for narr in df["complaint_narrative"]:
    sents = re.split(r'[.!?]+', narr)
    for s in sents:
        s = s.strip()
        if len(s) > 25:
            all_sentences.append(s)

sent_counter = Counter(all_sentences)

print("\nTop 15 repeated sentences across all narratives:")
print("-" * 70)
for sent, cnt in sent_counter.most_common(15):
    pct = cnt / len(df) * 100
    print(f"  [{cnt:6,} | {pct:5.1f}%] {sent[:90]}")

In [ ]:
# Quantify template severity
top_phrase = first_phrases.value_counts().head(1)
templates_using_top_starts = phrase_counts.head(5).sum()

print(f"\nTemplate Analysis Summary:")
print(f"  Most common opening covers {top_phrase.values[0]:,} complaints ({top_phrase.values[0]/len(df)*100:.1f}%)")
print(f"  Top 5 openings together cover {templates_using_top_starts:,} complaints ({templates_using_top_starts/len(df)*100:.1f}%)")
print(f"  Unique opening phrases: {len(phrase_counts):,}")
print(f"\n  WARNING: Narratives are heavily template-based.")
print(f"  Classifiers may learn template structure rather than genuine complaint semantics.")
print(f"  This limits generalization to real-world data.")

## 16. Issue vs Sentiment (Cross-Tabulation)

In [ ]:
issue_sentiment_ct = pd.crosstab(
    df["issue"], df["sentiment_label"],
    margins=True
)
print("Issue x Sentiment cross-tabulation:")
print(issue_sentiment_ct.to_string())

In [ ]:
# Heatmap — normalized by row (percentage within each issue)
issue_sentiment_pct = pd.crosstab(
    df["issue"], df["sentiment_label"], normalize="index"
) * 100

fig, ax = plt.subplots(figsize=(8, 10))
sns.heatmap(
    issue_sentiment_pct, annot=True, fmt=".1f", cmap="RdYlGn_r",
    cbar_kws={"label": "% of Issue"}, ax=ax
)
ax.set_title("Sentiment Distribution Within Each Issue (%)")
ax.set_ylabel("Issue")
ax.set_xlabel("Sentiment")
plt.tight_layout()
plt.show()

print("\nObservation: Sentiment ratios are roughly consistent across issues (~64% Neg, ~24% Neu, ~12% Pos).")
print("This suggests sentiment was generated independently of issue type.")

## 17. Resolution Status vs Sentiment

In [ ]:
res_sent_ct = pd.crosstab(df["resolution_status"], df["sentiment_label"])
print("Resolution Status x Sentiment:")
print(res_sent_ct.to_string())

print("\nKey observation:")
print("  - 'Unresolved' has ZERO Positive complaints.")
print("  - 'Resolved' has the highest Positive count.")
print("  - This is a strong synthetic signal — a model could infer resolution from sentiment.")

In [ ]:
fig = px.bar(
    df.groupby(["resolution_status", "sentiment_label"]).size().reset_index(name="count"),
    x="resolution_status", y="count", color="sentiment_label",
    barmode="group",
    title="Resolution Status by Sentiment",
    color_discrete_map={"Negative": "#e74c3c", "Neutral": "#f39c12", "Positive": "#27ae60"}
)
fig.show()

## 18. Channel and State Distribution

In [ ]:
channel_counts = df["channel"].value_counts()
print("Channel distribution:")
for ch, cnt in channel_counts.items():
    print(f"  {ch:12s}: {cnt:,} ({cnt/len(df)*100:.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Channel
axes[0].barh(channel_counts.index[::-1], channel_counts.values[::-1],
             color=sns.color_palette("Paired", len(channel_counts)))
axes[0].set_title("Complaints by Channel")
axes[0].set_xlabel("Count")

# Top 10 states
state_counts = df["state"].value_counts().head(10)
axes[1].barh(state_counts.index[::-1], state_counts.values[::-1],
             color=sns.color_palette("coolwarm", 10))
axes[1].set_title("Top 10 States by Complaint Volume")
axes[1].set_xlabel("Count")

plt.tight_layout()
plt.show()

print(f"\nNote: 'Other' state has {df[df['state']=='Other'].shape[0]:,} records — may represent synthetic catch-all.")

## 19. Product x Issue Breakdown

In [ ]:
prod_issue = df.groupby(["product", "issue"]).size().reset_index(name="count")
prod_issue_top = prod_issue.sort_values("count", ascending=False).head(20)

print("Top 20 Product–Issue combinations:")
for _, row in prod_issue_top.iterrows():
    print(f"  {row['product']:45s} | {row['issue']:25s} | {row['count']:,}")

In [ ]:
# Heatmap of product x issue
pivot = df.groupby(["product", "issue"]).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(pivot, annot=True, fmt="d", cmap="YlOrRd", linewidths=0.5, ax=ax)
ax.set_title("Product x Issue Complaint Counts")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 20. Data Quality Summary

In [ ]:
summary_lines = [
    "=" * 70,
    "EXPLORATORY DATA ANALYSIS — SUMMARY REPORT",
    "=" * 70,
    f"Dataset: data/raw/customer_complaints.csv",
    f"Generated: Synthetic dataset",
    "",
    "--- DIMENSIONS ---",
    f"Total rows:               {len(df):,}",
    f"Total columns:            {df.shape[1]}",
    f"Missing values:           0 (all columns complete)",
    f"Duplicate complaint IDs:  0",
    "",
    "--- NARRATIVE QUALITY ---",
    f"Unique narratives:        {df['complaint_narrative'].nunique():,} / {len(df):,}",
    f"Exact duplicate narratives: {df['complaint_narrative'].duplicated().sum():,} ({df['complaint_narrative'].duplicated().sum()/len(df)*100:.1f}%)",
    f"Character length range:   {df['narrative_length'].min()} – {df['narrative_length'].max()} (mean {df['narrative_length'].mean():.0f})",
    f"Template structure:       Heavy — most narratives follow fill-in-the-blank patterns",
    "",
    "--- CATEGORIES ---",
    f"Products:                 {df['product'].nunique()}",
    f"Issues:                   {df['issue'].nunique()}",
    f"Sub-issues:               {df['sub_issue'].nunique()}",
    f"Companies:                {df['company'].nunique()} (uniformly distributed — synthetic artifact)",
    "",
    "--- SENTIMENT ---",
    f"Negative:                 {(df['sentiment_label']=='Negative').sum():,} ({(df['sentiment_label']=='Negative').mean()*100:.1f}%)",
    f"Neutral:                  {(df['sentiment_label']=='Neutral').sum():,} ({(df['sentiment_label']=='Neutral').mean()*100:.1f}%)",
    f"Positive:                 {(df['sentiment_label']=='Positive').sum():,} ({(df['sentiment_label']=='Positive').mean()*100:.1f}%)",
    f"Class imbalance:          Moderate (Negative-dominant)",
    "",
    "--- DATE RANGE ---",
    f"From:                     {df['date_received'].min().date()}",
    f"To:                       {df['date_received'].max().date()}",
    "",
    "--- KEY OBSERVATIONS ---",
    "1. No missing data — complete synthetic generation.",
    "2. 28.8% exact-duplicate narratives — must group or deduplicate before train/test split.",
    "3. Narratives are template-based with shared filler sentences.",
    "4. Very narrow narrative length (std=13.7 chars) — unrealistic for real data.",
    "5. Sentiment is imbalanced (64% Negative) — use stratified splits.",
    "6. 'Unresolved' complaints have zero Positive sentiment — strong synthetic signal.",
    "7. Companies are uniformly distributed — unlike real-world patterns.",
    "8. Sentiment ratios are consistent across issues — likely generated independently.",
    "",
    "--- RECOMMENDATIONS FOR NEXT PHASES ---",
    "1. Use GroupKFold or narrative-hash grouping to prevent data leakage from duplicate texts.",
    "2. Report all classification metrics as 'synthetic-data performance'.",
    "3. Consider deduplicating narratives before embedding generation to save compute.",
    "4. Be cautious about resolution_status as a feature — it leaks sentiment information.",
    "5. Use stratified sampling for sentiment classification given class imbalance.",
    "=" * 70,
]

report_text = "\n".join(summary_lines)
print(report_text)

In [ ]:
# Save EDA report to file
report_path = os.path.join("..", "reports", "eda_report.txt")
if not os.path.exists(report_path):
    report_path = os.path.join("reports", "eda_report.txt")

os.makedirs(os.path.dirname(report_path), exist_ok=True)

with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_text)

print(f"\nEDA report saved to: {report_path}")

In [ ]:
# Clean up temporary columns before any downstream use
temp_cols = ["year", "day_of_week", "narrative_length", "narrative_word_count"]
df_clean = df.drop(columns=[c for c in temp_cols if c in df.columns])
print(f"Cleaned DataFrame shape: {df_clean.shape}")
print("Temporary analysis columns removed. Original 12 columns preserved.")

---

## End of EDA Notebook

**Next step:** Phase 2 — Issue Classification (`notebooks/03_issue_classification.ipynb`)

Before proceeding:
1. Review the charts above.
2. Decide on deduplication strategy for train/test splitting.
3. Confirm the issue/sub-issue label sets are acceptable for classification.